In [ ]:
library(Seurat)
# library(SeuratDisk)

library(reticulate)
library(anndata)

library(ggplot2)
library(ggpubr)
library(pheatmap)
library(dplyr)
library(tidyr)
library(RColorBrewer)
library(clustree)
library(repr)
options(repr.plot.width=10, repr.plot.height=8)

library(UpSetR)
library(grid)

library(PRROC)
library(Matrix)

library(reshape2)

getwd()

dataset_id <- "simulated_mm_RA_wGenes"
genome_id <- "mm10"
samples <- c("all")

dir.create("figures")
for (sample in samples) {
    dir.create(paste0("figures/figures_", dataset_id, "_", sample))
}
dir.create(paste0("figures/figures_", dataset_id))


colorTools <- c( # "MATES"="#F98A7B",
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)

thrMinCells <- 500 * 0.05

In [ ]:
load("workspaces/mm_RA_simulation_wGenes_evaluation_objectCreation.Rdata")

# Evaluation

## Upset plots of detected TEs and TPs

### Upsets of detected TEs

In [ ]:
options(repr.plot.width=12, repr.plot.height=7.6)

for (sample in samples){

    print(sample)

    TPlist <- list()
    FNlist <- list()
    FPlist <- list()

    detectedList <- list()

    for (tool in c(names(objList), "simulated")){

        print(tool)
        if (tool == "simulated") {
            obj <- splatter_objs[[sample]]
        }else {
            obj <- objList[[tool]][[sample]]
        }

        detectedList[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- Features(obj)
        TPlist[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- intersect(Features(splatter_objs[[sample]]), Features(obj))
        FNlist[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- setdiff(Features(splatter_objs[[sample]]), Features(obj))
        FPlist[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- setdiff(Features(obj), Features(splatter_objs[[sample]]))
    }

    pdf(file=paste0("figures/figures_", dataset_id, "_",sample,"/upset_detectedTEs.pdf"), width = 12, height = 7.6)
    show(UpSetR::upset(fromList(detectedList), nintersects = 15,
            sets=names(colorTools), keep.order = T, sets.bar.color=colorTools, 
            text.scale = c(2, 2, 2, 1.65, 2.5, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(detectedList))+1000, #show.numbers = F,
            nsets=length(objList)+1,
            sets.x.label="N. detected loci",
            mainbar.y.label="Intersection size")
        
    )
    grid.text(paste0("Detected TEs - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    dev.off()

    pdf(file=paste0("figures/figures_", dataset_id, "_",sample,"/upset_detected_TP.pdf"), width = 11, height = 7.5)
    show(UpSetR::upset(fromList(TPlist), nintersects = 15, 
            sets=names(colorTools), keep.order = T, sets.bar.color=colorTools, 
            text.scale = c(2, 2, 2, 1.65, 2.2, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(TPlist))+1000, #show.numbers = F, 
            nsets=length(c(objList, splatter_obj)),
            sets.x.label="N. correctly detected loci",
            mainbar.y.label="Intersection size")
    )
    grid.text(paste0("TP TEs - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))

    show(UpSetR::upset(fromList(FPlist), nintersects = 15, 
            sets=names(colorTools), keep.order = T, sets.bar.color=colorTools, 
            text.scale = c(2, 2, 2, 1.65, 2.2, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(FPlist))+1000, #show.numbers = F, 
            nsets=length(c(objList, splatter_obj)),
            sets.x.label="N. FP",
            mainbar.y.label="Intersection size")
    )
    grid.text(paste0("FP TEs - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))

    show(UpSetR::upset(fromList(FNlist), nintersects = 15, 
            sets=names(colorTools), keep.order = T, sets.bar.color=colorTools, 
            text.scale = c(2, 2, 2, 1.65, 2.2, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(FNlist))+1000, #show.numbers = F, 
            nsets=length(c(objList, splatter_obj)),
            sets.x.label="N. FN",
            mainbar.y.label="Intersection size")
    )
    grid.text(paste0("FN TEs - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    #|dev.off()
}


### Upsets of detected Genes

In [ ]:
options(repr.plot.width=12, repr.plot.height=7)

for (sample in samples){

    print(sample)

    TPgenelist <- list()
    FPgenelist <- list()
    FNgenelist <- list()
    detectedgeneList <- list()

    for (tool in c(names(objGeneList), "simulated")){

        print(tool)
        if (tool == "simulated") {
            obj <- splatter_objs_genes[[sample]]
        }else {
            obj <- objGeneList[[tool]][[sample]]
        }

        detectedgeneList[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- Features(obj)
        TPgenelist[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- intersect(Features(splatter_objs_genes[[sample]]), Features(obj))
        FNgenelist[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- setdiff(Features(splatter_objs_genes[[sample]]), Features(obj))
        FPgenelist[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- setdiff(Features(obj), Features(splatter_objs_genes[[sample]]))
    }

    options(repr.plot.width=12, repr.plot.height=6)
    pdf(file=paste0("figures/figures", "_", dataset_id, "_",sample,"/upset_detected.pdf"), width = 12, height = 6)
    show(UpSetR::upset(fromList(detectedgeneList), nintersects = 15,
            sets=names(colorTools[c("simulated", names(objGeneList))]), keep.order = T, sets.bar.color=colorTools[c("simulated", names(objGeneList))], 
            text.scale = c(2, 2, 2, 1.65, 2.5, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(detectedgeneList))+500, #show.numbers = F,
            nsets=length(objGeneList)+1,
            sets.x.label="N. detected genes",
            mainbar.y.label="Intersection size")
        
    )
    grid.text(paste0("Detected Genes - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    dev.off()

    pdf(file=paste0("figures/figures", "_", dataset_id, "_",sample,"/upset_detected_TP.pdf"), width = 9, height = 6)
    show(UpSetR::upset(fromList(TPgenelist), nintersects = 15, 
            sets=names(colorTools[c("simulated", names(objGeneList))]), keep.order = T, sets.bar.color=colorTools[c("simulated", names(objGeneList))], 
            text.scale = c(2, 2, 2, 1.65, 2.2, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(TPgenelist))+5000, #show.numbers = F, 
            nsets=length(c(objGeneList, splatter_obj)),
            sets.x.label="N. correctly detected Genes",
            mainbar.y.label="Intersection size")
    )
    grid.text(paste0("TP Genes - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    dev.off()

    options(repr.plot.width=9, repr.plot.height=6)
    pdf(file=paste0("figures/figures_", dataset_id, "_",sample,"/upset_detected_TP.pdf"), width = 9, height = 6)
    show(UpSetR::upset(fromList(FPgenelist), nintersects = 15, 
            sets=names(colorTools[c("simulated", names(objGeneList))]), keep.order = T, sets.bar.color=colorTools[c("simulated", names(objGeneList))], 
            text.scale = c(2, 2, 2, 1.65, 2.2, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(FPgenelist))+100, #show.numbers = F, 
            nsets=length(c(objGeneList, splatter_obj)),
            sets.x.label="N. FPGenes",
            mainbar.y.label="Intersection size")
    )
    grid.text(paste0("FP Genes - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    dev.off()
}


# TE - gene intresection

In [ ]:
library(rtracklayer)

gene_annotation <- rtracklayer::import("annotation/gencode.vM10.primary_assembly.annotation.gtf")
names(mcols(gene_annotation))

gene_annotation_df <- as.data.frame(gene_annotation)

# Extract only exon entries
exons <- gene_annotation[gene_annotation$type == "exon"]
names(exons) <- exons$gene_name

In [ ]:
# conversion_table <- read.table(paste0("annotation/annotation_", genome_id, "_conversion_withAge.tsv"))
# conversion_table

## Detected TEs overlapping exons

In [ ]:
overlapDfList_TE <- list()

# compute intersection between list of loci of each tool and gene annotation
for (sample in samples){
    overlapDfList_TE[[sample]] <- list()

    for(tool in names(objList)){

        print(tool)
        obj <- objList[[tool]][[sample]]
        bed_TE_tool <- conversion_table[conversion_table$stellarscopeID %in% Features(obj),c("chr","start","end","strand","stellarscopeID")]
        rownames(bed_TE_tool) <- bed_TE_tool$stellarscopeID
        # turn into GR object
        gr_TE_tool <- makeGRangesFromDataFrame(bed_TE_tool)

        #promOverlaps <- findOverlaps(gr_stellarscope, promoters, minoverlap = 10)
        exonOverlaps <- findOverlaps(gr_TE_tool, exons, minoverlap = 1)

        queryHits <- queryHits(exonOverlaps)
        subjectHits <- subjectHits(exonOverlaps)

        overlap_df <- data.frame(
            TE = names(gr_TE_tool)[queryHits],
            exon = names(exons)[subjectHits],
            gene_id = mcols(exons)$gene_id[subjectHits],
            transcript_id = mcols(exons)$transcript_id[subjectHits]
        )
        overlap_df$gene_name <- mcols(exons)$gene_name[subjectHits]

        # column indicating wether the overlapping gene is in the simulated gene matrix
        overlap_df$geneInSimulation <- overlap_df$gene_name %in% Features(splatter_objs_genes[[sample]])
        
        overlapDfList_TE[[sample]][[tool]] <- overlap_df
    }
}

In [ ]:
# compute intersection between list of loci of each tool and gene annotation
for (sample in samples){

    for(tool in names(objList)){

        print(tool)
        overlapDfList_TE[[sample]][[tool]]$detection <- NA

        overlapDfList_TE[[sample]][[tool]]$detection[overlapDfList_TE[[sample]][[tool]]$TE %in% TPlist[[tool]]] <- "TPs"
        overlapDfList_TE[[sample]][[tool]]$detection[overlapDfList_TE[[sample]][[tool]]$TE %in% FPlist[[tool]]] <- "FPs"        
        
    }
}

In [ ]:
percPosOverlapGenesDf <- NULL

for (sample in samples){

    for(tool in names(objList)){

        print(tool)
       
        df <- cbind(melt(table(overlapDfList_TE[[sample]][[tool]]$detection, overlapDfList_TE[[sample]][[tool]]$geneInSimulation )), tool)
        colnames(df) <- c("Detection", "ExpressedGene", "Number", "Tool")
        print(df)
        percPosOverlapGenesDf <- rbind(percPosOverlapGenesDf, df)

    }
}

# table(overlapDfList_TE[[sample]][[tool]]$detection)
# table(overlapDfList_TE[[sample]][[tool]]$detection, overlapDfList_TE[[sample]][[tool]]$geneInSimulation )


In [ ]:
percPosOverlapGenesDf$ExpressedGene <- as.character(percPosOverlapGenesDf$ExpressedGene)


In [ ]:
options(repr.plot.width=9, repr.plot.height=4.75)

ggplot(percPosOverlapGenesDf, aes(y=Tool, x=Number, fill=ExpressedGene)) +
    facet_wrap(~Detection) +
    geom_col(width = 0.85) +
    scale_fill_manual(values=c("TRUE"="#EC4D6A","FALSE"="#3d405b"),
                      labels = c("TRUE"="Expressed","FALSE"="Not"), 
                      breaks=c("TRUE", "FALSE"), name="which are") +
    theme_pubclean() +
    ggtitle("Detected TE loci intersecting genes") +
    xlab("Number of TE loci") +
  theme(text=element_text(size=20),
        axis.text.y = element_text(size=20),
        axis.label = element_text(size=18),
        legend.title = element_text(size=21),
        legend.text = element_text(size=21), 
        strip.text = element_text(size = 20),
        strip.background = element_rect(fill="grey90"),
        plot.title = element_text(size=21, hjust=0.5),
        plot.subtitle = element_text(size=20, hjust=0.5))
ggsave(paste0("figures/figures_", dataset_id, "_", sample,"/detectedTEsIntersectingGenes.pdf"), 
        device="pdf", width=9, height=4.5)

## Detected Genes overlapping TEs

In [ ]:
gc()

bed_TE <- conversion_table[,c("chr","start","end","strand","stellarscopeID")]
#rownames(bed_TE) <- bed_TE$stellarscopeID

gr_TE <- makeGRangesFromDataFrame(bed_TE, keep.extra.columns = TRUE )



In [ ]:
overlapDfList_GENE <- list()
overlapDfList_GENE_unique <- list()

# compute intersection between list of loci of each tool and gene annotation
for (sample in samples){
    overlapDfList_GENE[[sample]] <- list()

    for(tool in names(objGeneList)){

        print(tool)
        obj <- objGeneList[[tool]][[sample]]

        # get detected TE gr
        gr_GENE_tool <- exons[exons$gene_name %in% Features(obj)]

        
        # get detected TE gr
        # bed__tool <- conversion_table[conversion_table$stellarscopeID %in% Features(obj),c("chr","start","end","strand","stellarscopeID")]
        # rownames(bed_TE_tool) <- bed_TE_tool$stellarscopeID
        # # turn into GR object
        # gr_TE_tool <- makeGRangesFromDataFrame(bed_TE_tool)

        #promOverlaps <- findOverlaps(gr_stellarscope, promoters, minoverlap = 10)
        exonOverlaps <- findOverlaps(gr_GENE_tool, gr_TE, minoverlap = 1)

        queryHits <- queryHits(exonOverlaps)
        subjectHits <- subjectHits(exonOverlaps)

        overlap_df <- data.frame(
            Exon = names(gr_GENE_tool)[queryHits],
            gene_id = mcols(gr_GENE_tool)$gene_id[queryHits],
            gene_name = mcols(gr_GENE_tool)$gene_name[queryHits],
            TE = mcols(gr_TE)$stellarscopeID[subjectHits]
        )

        # column indicating wether the overlapping TE is in the simulated TE matrix
        overlap_df$TEInSimulation <- overlap_df$TE %in% Features(splatter_objs[[sample]])
        
        overlapDfList_GENE[[sample]][[tool]] <- overlap_df

        # create df with only the unique lines, prioritizing TEs that are in the simulation
        overlapDfList_GENE_unique[[sample]][[tool]]  <- overlapDfList_GENE[[sample]][[tool]] %>% distinct()
        overlapDfList_GENE_unique[[sample]][[tool]]  <- overlapDfList_GENE_unique[[sample]][[tool]]  %>%
                group_by(gene_name) %>%
                slice_max(order_by = TEInSimulation, n = 1, with_ties = FALSE) %>%
                ungroup()

    }
}


In [ ]:

## Detected genes overlapping TE loci
# compute intersection between list of loci of each tool and gene annotation
for (sample in samples){

    for(tool in names(objGeneList)){

        print(tool)
        overlapDfList_GENE_unique[[sample]][[tool]]$detection <- NA

        overlapDfList_GENE_unique[[sample]][[tool]]$detection[overlapDfList_GENE_unique[[sample]][[tool]]$gene_name %in% TPgenelist[[tool]]] <- "TPs"
        overlapDfList_GENE_unique[[sample]][[tool]]$detection[overlapDfList_GENE_unique[[sample]][[tool]]$gene_name %in% FPgenelist[[tool]]] <- "FPs"        
        
    }
}



In [ ]:

percPosOverlapTEsDf <- NULL

for (sample in samples){

    for(tool in names(objGeneList)){

        print(tool)
       
        df <- cbind(melt(table(overlapDfList_GENE_unique[[sample]][[tool]]$detection, overlapDfList_GENE_unique[[sample]][[tool]]$TEInSimulation )), tool)
        colnames(df) <- c("Detection", "ExpressedTE", "Number", "Tool")
        print(df)
        percPosOverlapTEsDf <- rbind(percPosOverlapTEsDf, df)

    }
}

# table(overlapDfList_TE[[sample]][[tool]]$detection)
# table(overlapDfList_TE[[sample]][[tool]]$detection, overlapDfList_TE[[sample]][[tool]]$geneInSimulation )


In [ ]:

percPosOverlapTEsDf$ExpressedTE <- as.character(percPosOverlapTEsDf$ExpressedTE)

options(repr.plot.width=8, repr.plot.height=4)

ggplot(percPosOverlapTEsDf, aes(y=Tool, x=Number, fill=ExpressedTE)) +
    facet_wrap(~Detection) +
    geom_col() +
    scale_fill_manual(values=c("TRUE"="#EC4D6A","FALSE"="#3d405b"),
                      labels = c("TRUE"="Expressed","FALSE"="Not"), 
                      breaks=c("TRUE", "FALSE"), name="which are") +
    theme_pubclean() +
    ggtitle("Detected Genes intersecting TEs") +
  theme(text=element_text(size=20),
        axis.text.y = element_text(size=18),
        axis.label = element_text(size=18),
        legend.title = element_text(size=18),
        legend.text = element_text(size=18), 
        strip.text = element_text(size = 20),
        strip.background = element_rect(fill="grey90"),
        plot.title = element_text(size=20, hjust=0.5))
ggsave(paste0("figures/figures_", dataset_id, "_", sample,"/detectedGenesIntersectingTEs.pdf"), device="pdf", width=8, height=5)

In [ ]:

percPosOverlapTEsDf$ExpressedTE <- as.character(percPosOverlapTEsDf$ExpressedTE)

options(repr.plot.width=7, repr.plot.height=5)

ggplot(percPosOverlapTEsDf[percPosOverlapTEsDf$Detection=="FPs",], aes(y=Tool, x=Number, fill=ExpressedTE)) +
    facet_wrap(~Detection) +
    geom_col() +
    scale_fill_manual(values=c("TRUE"="#EC4D6A","FALSE"="#3d405b"),
                      labels = c("TRUE"="Expressed","FALSE"="Not"), 
                      breaks=c("TRUE", "FALSE"), name="which are") +
    theme_pubclean() +
    ggtitle("Detected Genes intersecting TEs") +
  theme(text=element_text(size=20),
        axis.text.y = element_text(size=18),
        axis.label = element_text(size=18),
        legend.title = element_text(size=18),
        legend.text = element_text(size=18), 
        strip.text = element_text(size = 20),
        strip.background = element_rect(fill="grey90"),
        plot.title = element_text(size=20, hjust=0.5))
ggsave(paste0("figures/figures_", dataset_id, "_", sample,"/FPGenesIntersectingTEs.pdf"), device="pdf", width=6, height=5)

# Checkpoint

In [ ]:
#save.image("workspaces/mm_RA_wGenes_evaluation_detection_afterIntersections.Rdata")


In [ ]:
load("workspaces/mm_RA_wGenes_evaluation_detection_afterIntersections.Rdata")

## Precision / Recall - by Age


In [ ]:
layer <- "counts"

matSplatter <- GetAssayData(splatter_objs[[sample]], layer=layer)

# split the matrix by age
splatter_age_vec <- conversion_table$ageClass[match(rownames(matSplatter), conversion_table$stellarscopeID)]

matSplatter_age_List <- list()
matSplatter_age_List[["old"]] <- matSplatter[splatter_age_vec=="old",]
matSplatter_age_List[["young"]] <- matSplatter[splatter_age_vec=="young",]

precisionTool_age_List <- list()
recallTool_age_List <- list()

for(age in c("old", "young")){
  
  precisionTool_age_List[[age]] <- list()
  recallTool_age_List[[age]] <- list()
  
  for(tool in names(objList)){
  
    # get matrix of the tool, keep only TEs of the right age
    matTool <- as.matrix(GetAssayData(objList[[tool]][[sample]], layer=layer))
    
    print(table(rownames(matTool) %in% conversion_table$stellarscopeID))
    
    tool_age_vec <- conversion_table$ageClass[match(rownames(matTool), conversion_table$stellarscopeID)]
    matTool <- matTool[tool_age_vec==age,]
    
    precisionTool_age_List[[age]][[tool]] <- NULL
    recallTool_age_List[[age]][[tool]] <- NULL
    
    for(cell in Cells(splatter_objs[[sample]])){
      
      exprSplatter <- names(matSplatter_age_List[[age]][,cell])[matSplatter_age_List[[age]][,cell] > 0] # TEs expressed in the simulated matrix
      
      exprTool <- names(matTool[,cell])[matTool[,cell] > 0] # TEs detected by the tool

      TP <- intersect(exprSplatter, exprTool)
      nTP <- length(TP)
      
      FP <- setdiff(exprTool, exprSplatter)
      nFP <- length(FP)
      
      FN <- setdiff(exprSplatter, exprTool)
      nFN <- length(FN)
      
      precision <- nTP / (nTP + nFP)
      precisionTool_age_List[[age]][[tool]] <- c(precisionTool_age_List[[age]][[tool]], precision)
      
      recall <- nTP / (nTP + nFN)
      recallTool_age_List[[age]][[tool]] <- c(recallTool_age_List[[age]][[tool]], recall)
      
    }
  }
}


In [ ]:
options(repr.plot.width=5, repr.plot.height=3)

meanF1score_age_df <- NULL

for(age in c("old", "young")){
  
  precisionDf <- stack(precisionTool_age_List[[age]])
  colnames(precisionDf) <- c("precision", "tool")
  
  
  recallDf <- stack(recallTool_age_List[[age]])
  colnames(recallDf) <- c("recall", "tool")
  
  df <- merge(precisionDf, recallDf, by='tool')
  
  df$F1score <- (2 * df$precision * df$recall) / (df$precision + df$recall)
  
  
  df$tool <- factor(df$tool, levels=rev(names(colorTools)))
  
  show(
    ggplot(df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
      geom_boxplot(alpha = 0.5) + 
      scale_color_manual(values=colorTools) +
      scale_fill_manual(values=colorTools) +
      xlim(c(0,1)) +
      theme_light() + 
      ggtitle(paste0("F1-score - ", age)) +
      guides(fill="none", color="none") +
      theme(text=element_text(size=18), 
            plot.title = element_text(size=18, hjust=0.5), 
            plot.subtitle = element_text(size=18, hjust=0.5))
  )
#  ggsave(paste0("figures/figures_",dataset_id,"_",sample,"/F1score_tool_",age,".pdf"), device="pdf")
  
  show(
    ggplot(df, aes(x=precision, y=tool, color=tool, fill=tool)) + 
      geom_boxplot(alpha = 0.5) + 
      scale_color_manual(values=colorTools) +
      scale_fill_manual(values=colorTools) +
      xlim(c(0,1)) +
      theme_light() + 
      ggtitle(paste0("Precision - ", age)) +
      guides(fill="none", color="none") +
      theme(text=element_text(size=18), 
            plot.title = element_text(size=18, hjust=0.5), 
            plot.subtitle = element_text(size=18, hjust=0.5))
  )
#  ggsave(paste0("figures/figures_",dataset_id,"_",sample,"/precision_tool_",age,".pdf"), device="pdf")
  
  show(
    ggplot(df, aes(x=recall, y=tool, color=tool, fill=tool)) + 
      geom_boxplot(alpha = 0.5) +
      scale_color_manual(values=colorTools) +
      scale_fill_manual(values=colorTools) +
      xlim(c(0,1)) +
      theme_light() + 
      ggtitle(paste0("Recall - ", age)) +
      guides(fill="none", color="none") +
      theme(text=element_text(size=18), 
            plot.title = element_text(size=18, hjust=0.5), 
            plot.subtitle = element_text(size=18, hjust=0.5))
  )
#  ggsave(paste0("figures_",dataset_id,"_",sample,"/recall_tool_",age,".pdf"), device="pdf")
  
  
  # plot just the mean value
  
  precisionDf <- stack(lapply(precisionTool_age_List[[age]], mean))
  colnames(precisionDf) <- c("precision", "tool")

  recallDf <- stack(lapply(recallTool_age_List[[age]], mean))
  colnames(recallDf) <- c("recall", "tool")

  Fscores <- sapply(names(precisionTool_age_List[[age]]), function(tool){
                          (2 * precisionTool_age_List[[age]][[tool]] * recallTool_age_List[[age]][[tool]]) /
                          (precisionTool_age_List[[age]][[tool]] + recallTool_age_List[[age]][[tool]])
  })
  meanFscores <- colMeans(Fscores)
  FscoreDf <- stack(meanFscores)
  colnames(FscoreDf) <- c("F1score", "tool")

  df <- merge(precisionDf, recallDf, by=c("tool"))
  df <- merge(df, FscoreDf, by=c("tool"))

  df$tool <- factor(df$tool, levels=rev(names(colorTools)))
  
  # create df for facet plot
  meanF1score_age_df <- rbind(meanF1score_age_df, cbind(df, age)) 
  
  show(
    ggplot(df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
      geom_col(alpha = 0.9) +
      scale_color_manual(values=colorTools) +
      scale_fill_manual(values=colorTools) +
      xlim(c(0,1)) +
      theme_light() + ggtitle(paste0("Mean F1 score - ", age))  +
      guides(fill="none", color="none") +
      theme(text=element_text(size=18),
            plot.title = element_text(size=18, hjust=0.5),
            plot.subtitle = element_text(size=18, hjust=0.5))
    )
#  ggsave(paste0("figures_",dataset_id,"_",sample,"/meanF1score_tool_",age,".pdf"), device="pdf")

  show(
    ggplot(df, aes(x=precision, y=tool, color=tool, fill=tool)) +
      geom_col(alpha = 0.9) +
      scale_color_manual(values=colorTools) +
      scale_fill_manual(values=colorTools) +
      xlim(c(0,1)) +
      theme_light() + ggtitle(paste0("Mean precision - ", age))  +
      guides(fill="none", color="none") +
      theme(text=element_text(size=18),
            plot.title = element_text(size=18, hjust=0.5),
            plot.subtitle = element_text(size=18, hjust=0.5))
  )
#  ggsave(paste0("figures_",dataset_id,"_",sample,"/meanPrecision_tool_",age,".pdf"), device="pdf")

  show(
    ggplot(df, aes(x=recall, y=tool, color=tool, fill=tool)) +
      geom_col(alpha = 0.9) +
      scale_color_manual(values=colorTools) +
      scale_fill_manual(values=colorTools) +
      xlim(c(0,1)) +
      theme_pubclean() + ggtitle(paste0("Mean recall - ", age)) +
      guides(fill="none", color="none") +
      theme(text=element_text(size=18),
            plot.title = element_text(size=18, hjust=0.5),
            plot.subtitle = element_text(size=18, hjust=0.5))
  )
#  ggsave(paste0("figures_",dataset_id,"_",sample,"/meanRecall_tool_",age,".pdf"), device="pdf")

}

In [ ]:
precisionTool <- list()
recallTool <- list()

for (sample in samples){
    layer <- "counts"

    matSplatter <- GetAssayData(splatter_objs[[sample]], layer=layer)

    precisionTool[[sample]] <- list()
    recallTool[[sample]] <- list()

    for(tool in names(objList)){
        
        matTool <- GetAssayData(objList[[tool]][[sample]], layer=layer)
    
        precisionTool[[sample]][[tool]] <- NULL
        recallTool[[sample]][[tool]] <- NULL
        
        for(cell in Cells(splatter_objs[[sample]])){
            
            exprSplatter <- names(matSplatter[,cell])[matSplatter[,cell] > 0] 
            # TEs expressed in the simulated matrix
            
            exprTool <- names(matTool[,cell])[matTool[,cell] > 0] 
            # TEs detected by the tool

            TP <- intersect(exprSplatter, exprTool)
            nTP <- length(TP)
            
            FP <- setdiff(exprTool, exprSplatter)
            nFP <- length(FP)
            
            FN <- setdiff(exprSplatter, exprTool)
            nFN <- length(FN)
            
            precision <- nTP / (nTP + nFP)
            precisionTool[[sample]][[tool]] <- c(precisionTool[[sample]][[tool]], precision)
            
            recall <- nTP / (nTP + nFN)
            recallTool[[sample]][[tool]] <- c(recallTool[[sample]][[tool]], recall)    
        }
    }
}

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
 
ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 2) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1 score")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
#ggsave(paste0("figures/figures_",dataset_id,"_",sample,"/meanRecall_tool_byAge_horiz.pdf"), , height=3.2, width=8)


In [ ]:
options(repr.plot.width=5.5, repr.plot.height=7)

ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1 score")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
#ggsave(paste0("figures/figures_",dataset_id,"_",sample,"/meanF1_tool_byAge.pdf"), device="pdf")


ggplot(meanF1score_age_df, aes(x=precision, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean precision")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
#ggsave(paste0("figures/figures_",dataset_id,"_",sample,"/meanPrecision_tool_byAge.pdf"), device="pdf")

ggplot(meanF1score_age_df, aes(x=recall, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean recall")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
#ggsave(paste0("figures/figures_",dataset_id,"_",sample,"/meanRecall_tool_byAge.pdf"), device="pdf",width=5.5, repr.plot.height=7)


In [ ]:
options(repr.plot.width=9, repr.plot.height=6)

precision_plot <- ggplot(meanF1score_age_df, aes(x=precision, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean precision")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

recall_plot <- ggplot(meanF1score_age_df, aes(x=recall, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean recall")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

precision_plot + recall_plot
#ggsave(paste0("figures/figures_",dataset_id,"_",sample,"/meanPrecisionRecall_tool_byAge.pdf"), device="pdf", width=9, height=6)


In [ ]:
options(repr.plot.width=5, repr.plot.height=3)

for (sample in samples){
    print(sample)

    precisionDf <- stack(precisionTool[[sample]])
    colnames(precisionDf) <- c("precision", "tool")

    recallDf <- stack(recallTool[[sample]])
    colnames(recallDf) <- c("recall", "tool")

    df <- merge(precisionDf, recallDf, by='tool')

    df$F1score <- (2 * df$precision * df$recall) / (df$precision + df$recall)

    df$tool <- factor(df$tool, levels=rev(names(colorTools)))

    show(ggplot(df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
        geom_boxplot(alpha = 0.5) + 
        scale_color_manual(values=colorTools) +
        scale_fill_manual(values=colorTools) +
        theme_light() + ggtitle(paste0("F1-score - ", sample)) +
        guides(fill="none", color="none") +
        theme(text=element_text(size=18), 
                plot.title = element_text(size=18, hjust=0.5), 
                plot.subtitle = element_text(size=18, hjust=0.5))
    )

#    ggsave(paste0("figures/figures_",dataset_id,"_",sample,
 #       "/F1score_tool.png"))
    
    show(ggplot(df, aes(x=precision, y=tool, color=tool, fill=tool)) + 
        geom_boxplot(alpha = 0.5) + 
        scale_color_manual(values=colorTools) +
        scale_fill_manual(values=colorTools) +
        theme_light() + ggtitle(paste0("Precision - ", sample)) +
        guides(fill="none", color="none") +
        theme(text=element_text(size=18), 
                plot.title = element_text(size=18, hjust=0.5), 
                plot.subtitle = element_text(size=18, hjust=0.5))
    )
    #ggsave(paste0("figures/figures_",dataset_id,"_",sample,
     #   "/precision_tool.pdf"))

    show(ggplot(df, aes(x=recall, y=tool, color=tool, fill=tool)) + 
        geom_boxplot(alpha = 0.5) +
        scale_color_manual(values=colorTools) +
        scale_fill_manual(values=colorTools) +
        theme_light() + ggtitle(paste0("Recall - ", sample)) +
        guides(fill="none", color="none") +
        theme(text=element_text(size=18), 
                plot.title = element_text(size=18, hjust=0.5), 
                plot.subtitle = element_text(size=18, hjust=0.5))
    )

#    ggsave(paste0("figures/figures_",dataset_id,"_",sample,
 #       "/recall_tool.pdf"))

    # plot just the mean
    precisionDf <- stack(lapply(precisionTool[[sample]], mean))
    colnames(precisionDf) <- c("precision", "tool")

    recallDf <- stack(lapply(recallTool[[sample]], mean))
    colnames(recallDf) <- c("recall", "tool")

    Fscores <- sapply(names(precisionTool[[sample]]), function(tool){
                            (2 * precisionTool[[sample]][[tool]] * recallTool[[sample]][[tool]]) / 
                            (precisionTool[[sample]][[tool]] + recallTool[[sample]][[tool]])
    })
    meanFscores <- colMeans(Fscores)
    FscoreDf <- stack(meanFscores)
    colnames(FscoreDf) <- c("F1score", "tool")

    df <- merge(precisionDf, recallDf, by=c("tool"))
    df <- merge(df, FscoreDf, by=c("tool"))

    df$tool <- factor(df$tool, levels=rev(names(colorTools)))

    show(ggplot(df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
        geom_col(alpha = 0.9) + 
        scale_color_manual(values=colorTools) +
        scale_fill_manual(values=colorTools) +
        theme_light() + ggtitle(paste0("Mean F1-score - ", sample)) +
        guides(fill="none", color="none") +
        theme(text=element_text(size=18),
                plot.title = element_text(size=18, hjust=0.5),
                plot.subtitle = element_text(size=18, hjust=0.5))
    )
#    ggsave(paste0("figures/figures_",dataset_id,"_",sample,
 #       "/meanF1score_tool.pdf"))

    show(ggplot(df, aes(x=precision, y=tool, color=tool, fill=tool)) + 
        geom_col(alpha = 0.9) + 
        scale_color_manual(values=colorTools) +
        scale_fill_manual(values=colorTools) +
        theme_light() + ggtitle(paste0("Mean precision - ", sample)) +
        guides(fill="none", color="none") +
        theme(text=element_text(size=18), 
                plot.title = element_text(size=18, hjust=0.5), 
                plot.subtitle = element_text(size=18, hjust=0.5))
    )
#    ggsave(paste0("figures/figures_",dataset_id,"_",sample,
 #       "/meanPrecision_tool.pdf"))

    show(ggplot(df, aes(x=recall, y=tool, color=tool, fill=tool)) + 
        geom_col(alpha = 0.9) +
        scale_color_manual(values=colorTools) +
        scale_fill_manual(values=colorTools) +
        theme_light() + ggtitle(paste0("Mean recall - ", sample)) +
        guides(fill="none", color="none") +
        theme(text=element_text(size=18), 
                plot.title = element_text(size=18, hjust=0.5), 
                plot.subtitle = element_text(size=18, hjust=0.5))
    )
#    ggsave(paste0("figures/figures_",dataset_id,"_",sample,
 #       "/meanRecall_tool.pdf"))

}


In [ ]:
meanF1score_age_df <- NULL

for(age in c("old", "young")){
#   precisionDf <- stack(precisionTool[[age]])
#   colnames(precisionDf) <- c("precision", "tool")
  
  
#   recallDf <- stack(recallTool[[age]])
#   colnames(recallDf) <- c("recall", "tool")
  
#   df <- merge(precisionDf, recallDf, by='tool')
#   df$F1score <- (2 * df$precision * df$recall) / (df$precision + df$recall)
#   df$tool <- factor(df$tool, levels=rev(names(colorTools)))
  
  # plot just the mean value
  precisionDf <- stack(lapply(precisionTool[[age]], mean))
  colnames(precisionDf) <- c("precision", "tool")

  recallDf <- stack(lapply(recallTool[[age]], mean))
  colnames(recallDf) <- c("recall", "tool")

  Fscores <- sapply(names(precisionTool[[age]]), function(tool){
                          (2 * precisionTool[[age]][[tool]] * recallTool[[age]][[tool]]) /
                          (precisionTool[[age]][[tool]] + recallTool[[age]][[tool]])
  })
  
  meanFscores <- colMeans(Fscores)
  FscoreDf <- stack(meanFscores)
  colnames(FscoreDf) <- c("F1score", "tool")

  df <- merge(precisionDf, recallDf, by=c("tool"))
  df <- merge(df, FscoreDf, by=c("tool"))

  df$tool <- factor(df$tool, levels=rev(names(colorTools)))
  
  # create df for facet plot
  meanF1score_age_df <- rbind(meanF1score_age_df, cbind(df, age)) 

}


In [ ]:
options(repr.plot.width=8, repr.plot.height=4)

ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 2) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1 score")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
# ggsave(paste0("figures/figures_",dataset_id,"/meanF1_tool_byAge_horiz.pdf"), device="pdf", height=3.2, width=8)

In [ ]:
options(repr.plot.width=9, repr.plot.height=6)

precision_plot <- ggplot(meanF1score_age_df, aes(x=precision, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean precision")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

recall_plot <- ggplot(meanF1score_age_df, aes(x=recall, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean recall")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

precision_plot + recall_plot

#ggsave(paste0("figures/figures_",dataset_id,"/meanPrecisionRecall_tool_byAge.pdf"), 
 #     device="pdf", height=6, width=9)

splatter_objs
